Dado 6: Limites de Velocidade das Vias

* Fonte Oficial / Técnica: Referência regulamentar EPTC e Código de Trânsito Brasileiro (Art. 61); extração vetorial via OpenStreetMap (`maxspeed` e `highway`).
* Entrada: `dados_brutos/bruto_osm_malha_poa.geojson`
* Tratamento e Imputação Normativa:
  1. Extração Direta: Limpa e converte os valores numéricos da tag `maxspeed` cadastrados diretamente no OpenStreetMap (`OSM_Real`).
  2. Justificativa e Regra de Imputação (CTB): Como 70,2% dos trechos da malha viária não possuem sinalização de velocidade mapeada colaborativamente no OSM, aplicou-se o critério legal do **Art. 61 do Código de Trânsito Brasileiro** para vias urbanas sem sinalização: **80 km/h** para trânsito rápido (`trunk`), **60 km/h** para arteriais (`primary`), **40 km/h** para coletoras (`secondary`/`tertiary`) e **30 km/h** para vias locais (`residential`/`unclassified`), identificados como `Imputado_CTB`.
* Saída: `dados_limpos/limpo_06_velocidade_maxima.geojson`

In [6]:
import geopandas as gpd
import numpy as np

ruas = gpd.read_file("../../data/1_raw/bruto_osm_malha_poa.geojson")

# 1. Converte para texto antes de extrair os dígitos (preserva os ~26.350 valores reais do OSM)
ruas['velocidade_osm'] = ruas['maxspeed'].astype(str).str.extract(r'(\d+)').astype(float)

# 2. Limpa a hierarquia viária (sem usar colchetes para evitar erros de cópia)
texto_limpo = ruas['highway'].astype(str).str.replace(r"[\[\]']", "", regex=True)
hierarquia = texto_limpo.apply(lambda x: x.split(',')[0].strip())

regras_ctb = {
    'trunk': 80.0,
    'trunk_link': 60.0,
    'primary': 60.0,
    'primary_link': 60.0,
    'secondary': 40.0,
    'secondary_link': 40.0,
    'tertiary': 40.0,
    'tertiary_link': 40.0,
    'residential': 30.0,
    'living_street': 20.0,
    'unclassified': 30.0
}

velocidade_estimada_ctb = hierarquia.map(regras_ctb).fillna(30.0)

# 3. Combina: usa o dado real do OSM quando existir; aplica CTB nos demais
ruas['velocidade_max_kmh'] = ruas['velocidade_osm'].fillna(velocidade_estimada_ctb)
ruas['origem_velocidade'] = np.where(ruas['velocidade_osm'].notna(), 'OSM_Real', 'Imputado_CTB')

colunas = [c for c in ['u', 'v', 'key', 'name', 'velocidade_max_kmh', 'origem_velocidade', 'geometry'] if c in ruas.columns]
ruas[colunas].to_file("../../data/3_done/limpo_06_velocidade_maxima.geojson", driver="GeoJSON")

print(f"Concluído! 100% dos {len(ruas)} trechos agora possuem velocidade definida.")
print(ruas['origem_velocidade'].value_counts())

/Users/aluno-53/Documents/RIA26_Mobilidade_Urbana /seguranca_viaria_poa/venv/lib/python3.12/site-packages/geopandas/io/file.py:582: UserWarning: Could not parse column 'reversed' as JSON; leaving as string
  return pyogrio.read_dataframe(path_or_bytes, bbox=bbox, **kwargs)


Concluído! 100% dos 88347 trechos agora possuem velocidade definida.
origem_velocidade
Imputado_CTB    61997
OSM_Real        26350
Name: count, dtype: int64
